# Majority baseline

Run from `notebooks/` after `prepare_features.py`. Fit on 2020?2022 and evaluate on 2023.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))

from src.data.preprocess import notebook_root
from src.models.data import load_baseline_inputs
from src.models.majority_baseline import fit_baselines, evaluate_baselines, save_baseline_results

root = notebook_root()

## Load prepared features

Check artifact hashes, split years, labels, and matching rows for core and expanded.

In [ ]:
inputs = load_baseline_inputs(root)
pd.DataFrame({c: {split: item[split][0].shape for split in ('train', 'validation')}
              for c, item in inputs.items()})

## Fit the majority class

Each model predicts the most frequent training label, without sample weights.

In [ ]:
models = fit_baselines(inputs)
{c: dict(zip(model.classes_, model.class_prior_)) for c, model in models.items()}

## Evaluate validation predictions

Both feature sets should give identical predictions because this baseline ignores predictor values.

In [ ]:
results = evaluate_baselines(inputs, models)
pd.DataFrame([result['metrics'] for result in results.values()])

In [ ]:
results['core']['per_class']

In [ ]:
results['core']['confusion'].pivot(index='actual', columns='predicted', values='rows')

## Save results

Small reports go in `results/majority_baseline/`; crash-level predictions stay local in `data/processed/predictions/majority_baseline/`.

In [ ]:
save_baseline_results(root, inputs, results)